# 03 — Preprocessing: Home Credit Default Risk

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import os

RAW_PATH = '../data/raw/home_credit/application_train.csv'
OUT_PATH = '../data/processed/v4/home_credit.csv'

os.makedirs('../data/processed/v4', exist_ok=True)

## 1. Load and Basic Checks

In [ ]:
df = pd.read_csv(RAW_PATH)
print(f'Shape: {df.shape}')
print(f'TARGET distribution:')
print(df['TARGET'].value_counts())
print(f'Default rate: {df["TARGET"].mean():.4f}  ({df["TARGET"].sum()} defaults of {len(df)})')

df = df.drop(columns=['SK_ID_CURR'])

df = df.rename(columns={'TARGET': 'target'})
print(f'\nShape after ID drop: {df.shape}')

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Numeric columns: {len(num_cols)}')
print(f'Categorical columns: {len(cat_cols)}')
print(f'\nCategorical columns: {cat_cols}')

## 2. DAYS_EMPLOYED Anomaly

In [ ]:
n_anomaly = (df['DAYS_EMPLOYED'] == 365243).sum()
print(f'DAYS_EMPLOYED == 365243: {n_anomaly} cases ({n_anomaly/len(df):.1%})')

df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)

df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

print(f'DAYS_EMPLOYED after fix:')
print(df['DAYS_EMPLOYED'].describe().round(1))
print(f'DAYS_EMPLOYED_ANOM Flag: {df["DAYS_EMPLOYED_ANOM"].sum()} gesetzt')

## 3. Missingness Profile (Reported Only)

In [ ]:
from src.datasets import MISSING_THRESHOLD

missing_frac = df.isnull().mean()
above = sorted(missing_frac[missing_frac > MISSING_THRESHOLD].index.tolist())

print(f'Columns above {MISSING_THRESHOLD:.0%} missing (kept in the export, filter runs '
      f'pro Fit-Subset): {len(above)}')
for c in above:
    print(f'  {c}: {missing_frac[c]:.1%}')
print(f'\nShape unchanged: {df.shape}')

## 4. XNA Values to NaN

In [ ]:
cat_cols_current = df.select_dtypes(include='object').columns.tolist()
xna_counts = {}
for col in cat_cols_current:
    n = (df[col] == 'XNA').sum()
    if n > 0:
        xna_counts[col] = n

print('Columns with XNA values:')
for col, n in xna_counts.items():
    print(f'  {col}: {n}')

for col in cat_cols_current:
    df[col] = df[col].replace('XNA', np.nan)

## 5. AMT_INCOME_TOTAL — Outlier Profile (No Clipping Here)

In [ ]:
p99_reference = df['AMT_INCOME_TOTAL'].quantile(0.99)
print(f'AMT_INCOME_TOTAL: max={df["AMT_INCOME_TOTAL"].max():,.0f}, '
      f'P99 (reference only, not applied)={p99_reference:,.0f}, '
      f'above: {(df["AMT_INCOME_TOTAL"] > p99_reference).sum()} values')

## 6. Categorical Columns — Kept Raw

In [ ]:
from src.datasets import get

config = get('home_credit')
declared = list(config.categorical_cols)
present = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]

assert sorted(present) == sorted(declared), (
    f'Categorical columns deviate from src/datasets.py:\n'
    f'  im Frame:    {sorted(present)}\n'
    f'  deklariert:  {sorted(declared)}'
)
print(f'{len(declared)} categorical columns kept raw:')
for c in declared:
    print(f'  {c}: {df[c].nunique(dropna=True)} Level, {df[c].isnull().mean():.1%} fehlend')

In [ ]:
missing_by_categorical = {
    c: int(df[c].isna().sum()) for c in config.categorical_cols if df[c].isna().any()
}
print('Fehlwerte je kategorialer Spalte (bleiben erhalten, Pipeline setzt "Unknown"):')
for c, n in sorted(missing_by_categorical.items(), key=lambda kv: -kv[1]):
    print(f'  {c}: {n} ({n / len(df):.1%})')
print(f'\nShape unchanged: {df.shape}')

## 7. Missing Values Kept

In [ ]:
missing_before = df.isnull().sum()
missing_cols = missing_before[missing_before > 0]
print(f'Columns with missing values (kept as NaN, imputed per fit subset): {len(missing_cols)}')
print(missing_cols.sort_values(ascending=False).head(20))
print(f'\nGesamt-Fehlwerte (erwartet > 0): {df.isnull().sum().sum()}')
print(f'Target-Fehlwerte (muss 0 sein): {df["target"].isnull().sum()}')

## 8. Sanity Checks

In [ ]:
assert df['target'].isnull().sum() == 0, 'Target has missing values'
n_missing = df.drop(columns=['target']).isnull().sum().sum()
print(f'✓ Target ohne Missings — Feature-Missings (Pipeline imputiert): {n_missing}')
assert n_missing > 0, 'Expected: missing values survive the export'

non_num = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
assert sorted(non_num) == sorted(config.categorical_cols), \
    f'Unexpected non-numeric columns: {sorted(set(non_num) ^ set(config.categorical_cols))}'
print(f'✓ {len(non_num)} categorical columns raw, all others numeric')

assert set(df['target'].unique()) == {0, 1}, 'Target is not binary'
print(f'✓ Target binary — default rate: {df["target"].mean():.4f}')

n_neg = (df['target'] == 0).sum()
n_pos = (df['target'] == 1).sum()
print(f'  Classes: {n_neg} (no default) vs {n_pos} (default) → ratio {n_neg / n_pos:.1f}:1')
print('  Imbalance via scale_pos_weight per fit call — no oversampling')

print(f'\nFinaler Shape: {df.shape}')

In [ ]:
key_cols = ['AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY', 'DAYS_BIRTH',
            'DAYS_EMPLOYED', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
key_cols = [c for c in key_cols if c in df.columns]
df[key_cols].describe().round(2)

## 9. Save

In [ ]:
df = df.copy()
df.insert(0, 'source_row_id', np.arange(len(df), dtype=np.int64))
cols = [c for c in df.columns if c != 'target'] + ['target']
df = df[cols]

df.to_csv(OUT_PATH, index=False)
print(f'Gespeichert: {OUT_PATH}')
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
from src.semiraw import audit_semiraw_export

report = audit_semiraw_export(OUT_PATH, 'home_credit', expect_missing=True)